<div align="center">
    <h2><b>Architecture Design Document (ADD)</b></h2>
    <h1>Інтеграція й оптимізація комунікації в складній розподіленій системі</h1>
    <h3>Платформа бронювання авіаквитків «2BeFlüge»</h3>
    <br>

**Метадані документа:**

| **Тип документа:** | High-Level Design (HLD) & Communication Strategy |
| :--- | :--- |
| **Статус:** | Final / Release |
| **Версія архітектури:** | 1.0.5 |
| **Дата створення:** | Березень 2026 р. |

<br>

**Автор проекту:**

| **Ім'я:** | Олег Гаценко |
| :--- | :--- |
| **Роль:** | System Architect / Студент магістратури |
| **Організація:** | NeoVersity (Master's in AI & Machine Learning) |

</div>

---

### **Executive Summary (Резюме архітектури):**

Цей документ описує комплексне проєктування високорівневої архітектури (HLD) для платформи бронювання авіаквитків **2BeFlüge**, розрахованої на роботу в умовах екстремальних навантажень (B2C Флеш-розпродажі, Black Friday). Оскільки система працює з фінансовими транзакціями, інвентаризацією місць у реальному часі та персоналізованими пропозиціями, базовим підходом було обрано гібридну мікросервісну архітектуру з акцентом на подійно-орієнтовану модель (Event-Driven Architecture).

**Ключові архітектурні парадигми та рішення:**
* **Hybrid Communication & Elasticity:** Чітке розділення на синхронне ядро (gRPC) для швидких транзакцій та подійно-орієнтовану шину (RabbitMQ/Kafka) для асинхронних процесів. Динамічне масштабування (HPA) гарантує стабільність для веб-клієнтів та **мобільних застосунків**.
* **Resiliency & Chaos Engineering:** Ізоляція збоїв та захист від каскадних відмов за допомогою патернів `Circuit Breaker` (із джиттером), `Bulkhead` та `Dead Letter Queue (DLQ)`. Працездатність закладених рішень безперервно верифікується методологією хаос-інженерії (Chaos Mesh).
* **Distributed Transactions:** Відмова від повільних та нестабільних розподілених блокувань (2PC) на користь компенсаційних транзакцій (**Saga Pattern**) та ключів ідемпотентності, що на 100% гарантує цілісність розірваних фінансових даних.
* **Zero-Downtime Evolution:** Безшовна еволюція системи завдяки патерну Anti-Corruption Layer на API Gateway, стандарту помилок RFC 7807, безпечним міграціям БД (Expand and Contract) та математичній моделі Sunsetting для старих клієнтів.
* **ML Integration & Smart Cache:** Кешування в Redis для віддачі персоналізованих пропозицій за час $O(1)$ та безперервне A/B тестування моделей за алгоритмом *Multi-Armed Bandit*.

### **Глобальна архітектурна схема (Високорівнева):**

```mermaid
flowchart TD
    Client([Web / Mobile<br/>Клієнти 2BeFlüge])
    Gateway{{API Gateway / GraphQL}}

    Client -->|API запити| Gateway

    subgraph Core ["Синхронне&nbsp;ядро&nbsp;(Low&nbsp;Latency)"]
        Booking[Сервіс бронювання]
        Inventory[Сервіс інвентарю]
        Payment[Платіжний сервіс]
    end

    Gateway -->|GraphQL / REST| Booking
    Booking <-->|gRPC| Inventory
    Booking <-->|gRPC| Payment
    Gateway -->|REST| Payment

    Broker[[Apache Kafka Broker]]

    Booking -.->|Event: BookingCreated| Broker
    Payment -.->|Event: PaymentProcessed| Broker

    subgraph Async ["Асинхронні&nbsp;підсистеми&nbsp;(Event&nbsp;Driven)"]
        Notification[Сервіс сповіщень]
        Analytics[Сервіс аналітики]
        Recommendation[Сервіс ML рекомендацій]
    end

    Broker -.->|Pub/Sub| Notification
    Broker -.->|Pub/Sub| Analytics
    Broker -.->|Pub/Sub| Recommendation

    classDef sync fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000
    classDef async fill:#fff8e1,stroke:#ff8f00,stroke-width:2px,stroke-dasharray: 5 5,color:#000
    classDef broker fill:#eceff1,stroke:#37474f,stroke-width:3px,color:#000

    class Booking,Inventory,Payment sync
    class Notification,Analytics,Recommendation async
    class Broker broker

    style Core fill:#ffea00,stroke:#1565c0,stroke-width:1px
    style Async fill: #009dff,stroke:#ff8f00,stroke-width:1px
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **1. Проектування комунікації між мікросервісами**

У високонавантаженій системі бронювання авіаквитків **2BeFlüge** вибір протоколу взаємодії залежить від того, чи знаходиться сервіс на **Критичному шляху** (Critical Path) користувача. 

Ми застосовуємо гібридний підхід: **Синхронна комунікація (gRPC/REST)** використовується виключно там, де необхідна строга консистентність даних (запобігання подвійному бронюванню), а **Асинхронна (Event-Driven)** — для масштабування побічних бізнес-процесів і захисту від каскадних відмов під час флеш-розпродажів.

### 1.1. Матриця взаємодії компонентів (Communication Matrix)

| Відправник | Отримувач | Тип зв'язку | Технологія | Обґрунтування (Trade-offs) |
| :--- | :--- | :--- | :--- | :--- |
| **Клієнт** (Web/Mobile) | **API Gateway** | Синхронний | **GraphQL** | Мобільні клієнти часто мають нестабільне з'єднання. GraphQL усуває проблеми *Overfetching* (завантаження зайвих даних) та *Underfetching* (необхідність робити 5 запитів замість 1). Клієнт сам формує структуру відповіді (маршрут + ціна + наявність місць). |
| **API Gateway** | **Сервіс бронювання** | Синхронний | **REST API** | Використовуємо REST (JSON через HTTP) як стандартний, легко кешований протокол для зовнішньої маршрутизації. Це спрощує налаштування Rate Limiting та WAF на рівні Gateway. |
| **Сервіс бронювання** | **Сервіс інвентарю** | Синхронний | **gRPC** | **Критично низька затримка.** Перевірка і резервування місця в літаку вимагає жорсткої консистентності (ACID). Бінарний формат Protobuf швидший за JSON у 5-7 разів, а строгі контракти усувають помилки типізації між сервісами. |
| **Сервіс бронювання** | **Платіжний сервіс** | Синхронний | **gRPC** | Ініціалізація транзакції та холдування (блокування) коштів вимагає надійного RPC-виклику. gRPC підтримує вбудовані дедлайни (Timeouts) та механізми скасування, що критично для фінансових операцій. |
| **Будь-який сервіс** | **Сервіс сповіщень** | Асинхронний | **RabbitMQ** (Message Queue) | Користувач не повинен чекати, поки SMTP-сервер надішле email. Використовуємо патерн *Point-to-Point* (черга задач). RabbitMQ ідеально підходить для гарантованої доставки (At-least-once) та має вбудований механізм Dead-Letter Queues (DLQ) для листів, що не відправилися. |
| **Ядро** (Бронювання, Оплати) | **Аналітика та Рекомендації (ML)** | Асинхронний | **Apache Kafka** (Event Stream) | Патерн *Pub/Sub*. Сервіси ядра публікують події (напр., `FlightSearched`, `TicketPurchased`). Аналітика і ML вичитують їх незалежно у реальному часі. Kafka обрана через здатність витримувати мільйони подій за секунду та можливість "перемотувати" історію подій, якщо ML-сервіс тимчасово впаде. |

**Як підсумок:** обраний гібридний стек гарантує блискавичну швидкодію та транзакційну цілісність критичного ядра, повністю ізолюючи його від важких фонових завдань, що дозволяє системі гнучко та незалежно масштабуватися.

---

### 1.2. Аналіз компромісів та сценарій «Флеш-розпродаж» (Flash Sale)

Під час акцій навантаження на систему зростає експоненціально. Якщо всі комунікації залишити синхронними, Платіжний сервіс (або зовнішній банківський шлюз) стане вузьким місцем (Bottleneck), що призведе до вичерпання пулу з'єднань і каскадного падіння всієї платформи.

**Шляхи оптимізації комунікації під час піків:**
1. **Зворотний тиск (Backpressure):** Замість того, щоб чекати синхронної відповіді від банку, API Gateway приймає запит на оплату, миттєво повертає клієнту HTTP статус `202 Accepted` ("Опрацьовується") і скидає завдання у швидку чергу (Kafka).
2. **Асинхронна обробка:** Воркери Платіжного сервісу забирають повідомлення з черги з тією швидкістю, яку фізично може витримати банківський API. 
3. **Event-Driven UI:** Клієнтський додаток (через WebSockets або Server-Sent Events) отримує асинхронне сповіщення про успішну покупку щойно Платіжний сервіс завершить транзакцію і опублікує подію `PaymentSuccess`.

**Візуалізація асинхронного потоку оплати (Sequence Diagram):**

```mermaid
sequenceDiagram
    autonumber
    actor Client as Клієнт (UI)
    participant GW as API Gateway
    participant Broker as Kafka (Черга)
    participant Worker as Payment Worker
    participant Bank as Банківський Шлюз

    Note over Client, Bank: Флеш-розпродаж (Пікове навантаження)

    Client->>GW: POST /api/v1/payments (Запит на оплату)
    GW->>Broker: Публікація події [PaymentRequested]
    Broker-->>GW: ACK (Подію успішно збережено)
    
    %% Швидка відповідь клієнту без блокування
    GW-->>Client: HTTP 202 Accepted (Платіж в обробці)
    Note left of Client: UI показує спіннер<br/>"Обробляємо оплату..."

    %% Асинхронна обробка в бекграунді
    Broker->>Worker: Консьюмер вичитує подію з черги
    Worker->>Bank: Синхронний виклик (в межах лімітів банку)
    Bank-->>Worker: Відповідь (Успіх)

    Worker->>Broker: Публікація події [PaymentSuccess]
    Note right of Broker: Notification Service вичитує<br/>подію і пушить клієнту
```

**Математичне обґрунтування (Закон Літтла):**

Фундаментальною причиною для переходу на асинхронні черги є класична теорія масового обслуговування (Queueing Theory), зокрема Закон Літтла:

$$L = \lambda \cdot W$$

Де:
* $L$ — кількість одночасних запитів у системі (наш пул з'єднань на Gateway).
* $\lambda$ — швидкість надходження нових запитів від користувачів.
* $W$ — середній час обробки одного запиту (обмежений повільним API банку).

Під час флеш-розпродажу параметр $\lambda$ зростає експоненціально. Оскільки ми фізично не можемо зменшити $W$ (швидкість зовнішнього банку є константою), параметр $L$ неминуче проб'є стелю лімітів нашого API Gateway. **Єдиний математично вірний спосіб** уникнути відмови всієї системи — це розірвати синхронний ланцюг і перенести $L$ у високоємну буферну чергу.

---

### 1.3. Неочевидні аспекти та еволюція технологій (Replaceability)

Вибір гібридного стека комунікацій тягне за собою кілька неочевидних архітектурних наслідків:

1. **Вплив на клієнтські додатки (UI/UX):** Перехід на асинхронну обробку (особливо оплат) вимагає зміни парадигми на фронтенді. Замість простого HTTP-запиту з очікуванням відповіді, клієнтські додатки мають підтримувати Event-Driven підхід (наприклад, піднімати **WebSocket**-з'єднання або використовувати **Server-Sent Events**), щоб слухати сповіщення від системи, коли бекграунд-процес завершиться.
2. **Складність моніторингу (Observability):** У розподіленій системі, де запит стрибає з REST у gRPC, а потім у Kafka, класичне логування стає неефективним. Це вимагає обов'язкового впровадження систем **Distributed Tracing** (наприклад, Jaeger або OpenTelemetry). Кожному початковому запиту на API Gateway присвоюється унікальний `Correlation-ID`, який передається у заголовках (Headers) або метаданих усіх наступних gRPC-викликів та повідомлень Kafka.
3. **Можливість заміни технологій (Replaceability):**
   - Завдяки використанню **API Gateway**, зовнішні клієнти повністю ізольовані від внутрішньої топології. Якщо в майбутньому ми вирішимо замінити REST на gRPC між Booking Service та іншими новими модулями, публічні клієнти цього навіть не помітять.
   - Для брокерів повідомлень застосовується патерн **Ports and Adapters (Гексагональна архітектура)**. Бізнес-логіка мікросервісу нічого не знає про специфіку RabbitMQ чи Kafka. Вона працює через абстрактний інтерфейс `MessagePublisher`. Якщо пропускної здатності RabbitMQ для `Notification Service` стане недостатньо, інженери зможуть замінити його на топік Kafka, переписавши лише один клас-адаптер, без жодного втручання в логіку формування самих сповіщень.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **2. Оптимізація продуктивності та масштабованості (Платіжний сервіс)**

Під час флеш-розпродажів (Flash Sales) або сезонних сплесків трафік може зростати в десятки разів за лічені хвилини. Головним «вузьким місцем» (bottleneck) зазвичай стає Платіжний сервіс, оскільки він залежить від пропускної здатності зовнішніх банківських шлюзів (які часто працюють повільно і мають власні жорсткі ліміти на кількість запитів).

Щоб запобігти вичерпанню пулу з'єднань і каскадному падінню системи, ми переводимо Платіжний сервіс на **асинхронну архітектуру з використанням черг повідомлень**.

### 2.1. Архітектура Платіжного сервісу (Payment System Design)

```mermaid
flowchart TD
    Client([Клієнт 2BeFlüge]) -->|POST /api/v1/payments| Gateway

    subgraph Gateway_Tier ["API&nbsp;Gateway&nbsp;Tier"]
        Gateway{{API Gateway}}
        RateLimiter[(Rate Limiter / Redis)]
        Gateway <-->|Перевірка лімітів| RateLimiter
    end

    Gateway -->|HTTP 202 Accepted| PaymentAPI

    subgraph Payment_System ["Payment&nbsp;Service&nbsp;(Kubernetes)"]
        PaymentAPI[Payment API Pods]
        Cache[(Redis Cache<br/>Ключі ідемпотентності)]
        Queue[[RabbitMQ / SQS<br/>Черга транзакцій]]
        Worker[Payment Workers<br/>Автомасштабування]

        PaymentAPI <-->|Перевірка дублів| Cache
        PaymentAPI -->|Публікація задачі| Queue
        Queue -->|Вичитування задачі| Worker
    end

    Worker <-->|Синхронний API виклик| Bank((Зовнішній<br/>Банківський API))
    Worker -.->|Event: PaymentSuccess| Kafka[[Apache Kafka<br/>Event Bus]]

    classDef sys fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000
    classDef external fill:#ffebee,stroke:#c62828,stroke-width:2px,color:#000
    class PaymentAPI,Cache,Queue,Worker sys
    class Bank external
```

---

### 2.2. Ключові механізми оптимізації та захисту

Для забезпечення безперебійної роботи ми впроваджуємо наступні патерни:

1. **Обмеження швидкості (Rate Limiting) на API Gateway:**
   Використовуємо алгоритм *Token Bucket*. Gateway обмежує кількість запитів на оплату від одного користувача/IP (наприклад, не більше 3 спроб на хвилину). Це захищає систему від DDoS-атак та випадкового спаму кнопкою «Оплатити».
2. **Асинхронна обробка запитів (Message Queues):**
   Замість того, щоб тримати HTTP-з'єднання з клієнтом відкритим, поки банк обробляє платіж (що може тривати 5-10 секунд), `Payment API` лише приймає запит, валідує його, кладе у внутрішню чергу транзакцій (RabbitMQ) і миттєво повертає клієнту статус `202 Accepted` ("Платіж в обробці"). Клієнтський UI переходить у стан очікування.
3. **Автоматичне масштабування (Auto-scaling) на основі черги:**
   Ми налаштовуємо *Kubernetes Horizontal Pod Autoscaler (HPA)*. Метрикою для масштабування є не використання CPU, а **довжина черги (Queue Length)**. Якщо в черзі RabbitMQ накопичується більше 1000 необроблених платежів, Kubernetes автоматично піднімає додаткові контейнери `Payment Workers`, щоб швидше розгрібати транзакції. 
4. **Кешування та Ідемпотентність (Idempotency):**
   Під час флеш-розпродажів користувачі часто панікують і натискають кнопку оплати кілька разів, якщо екран довго вантажиться. Щоб уникнути подвійного списання коштів, клієнт генерує унікальний `Idempotency-Key` для кожної спроби. `Payment API` зберігає цей ключ у швидкому кеші (Redis). Якщо надходить дублікат запиту, система відкидає його ще до потрапляння в чергу.
5. **Зворотний тиск (Backpressure) на воркери:**
   Навіть якщо ми піднімемо 100 воркерів, зовнішній банк може не витримати такого потоку. Воркери налаштовуються на вичитування повідомлень з черги рівно з тією швидкістю, яку дозволяє ліміт банку (наприклад, 50 транзакцій на секунду). Усі інші запити безпечно чекають своєї черги в RabbitMQ.

---

### 2.3. Неочевидні аспекти, вузькі місця та еволюція (Replaceability)

Впровадження асинхронної платіжної системи створює нові архітектурні виклики та можливості для системи в цілому:

1. **Вплив на систему бронювання (Distributed Locks):**
Оскільки оплата відбувається асинхронно, сервіс Бронювання (Booking Service) змушений імплементувати стан `PAYMENT_PENDING`. Місце в літаку тимчасово блокується (Soft Lock) на 15 хвилин. Якщо воркер не публікує подію `PaymentSuccess` протягом цього часу, запускається фоновий процес (TTL-індекс у базі даних), який автоматично знімає блокування і повертає квиток у вільний продаж.
2. **Життєвий цикл транзакції (State Machine):**

**Для коректного трекінгу оплат вводиться строгий автомат станів:**

```mermaid
stateDiagram-v2
    [*] --> PENDING : HTTP 202 (Запит у черзі)
    PENDING --> PROCESSING : Воркер взяв у роботу
    PROCESSING --> SUCCESS : Банк підтвердив
    PROCESSING --> FAILED : Банк відхилив /<br/>Немає коштів
    PROCESSING --> PENDING : Мережева помилка (Retry)

    SUCCESS --> [*]
    FAILED --> [*]
```

3. **Змінність технологій (Replaceability):**
	- **Черги:** Логіка воркерів не прив'язана до конкретного брокера. Якщо інфраструктура мігрує в екосистему AWS, ми можемо легко замінити RabbitMQ на керований `Amazon SQS` без переписування ядра системи обробки платежів.
	- **Провайдери:** Самі зовнішні банківські шлюзи (Stripe, PayPal, Braintree) сховані за патерном **Adapter (Адаптер)**. Додавання нового методу оплати не впливає на структуру черг чи API — інженерам потрібно лише написати новий клас-адаптер для інтеграції з новим банком.
	- **Кеш (Single Point of Failure):** Якщо Redis, який тримає Rate Limits та ключі ідемпотентності, не витримає навантаження, ми можемо перевести його в режим **Redis Cluster** з партиціюванням (Sharding) ключів по `user_id`, розподіливши навантаження на кілька фізичних серверів.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **3. Розробка стратегії обробки помилок (Error Handling & Resiliency)**

У мікросервісній архітектурі ми виходимо з аксіоми: **сервіси будуть падати**. Наша мета — не запобігти всім збоям, а ізолювати їх за допомогою патернів відмовостійкості, гарантуючи **Graceful Degradation (м'яку деградацію)**. Користувач повинен завжди отримувати прогнозовану відповідь, навіть якщо частина системи недоступна.

### 3.1. Захист синхронного ядра (Circuit Breaker & Retry with Jitter)

**Сценарій А: Сервіс Інвентарю (Inventory) недоступний під час бронювання**

- **Проблема:** Сервіс `Booking Service` не може зв'язатися з `Inventory Service` через мережевий таймаут. Прості, неконтрольовані повторні спроби (Retries) можуть створити шквал запитів і "добити" сервіс, що намагається відновитися.
- **Рішення (Патерн Circuit Breaker):** Ми встановлюємо Circuit Breaker (Запобіжник) на клієнті gRPC. Він працює як автомат станів:

```mermaid
stateDiagram-v2
    direction LR
    state "CLOSED (🟢 Норма)" as Closed
    state "OPEN (🔴 Збій)" as Open
    state "HALF-OPEN (🟡 Перевірка)" as HalfOpen

    Closed --> Open : Відсоток помилок > 50%
    Open --> HalfOpen : Таймаут очікування (напр. 30 сек)
    HalfOpen --> Closed : Наступний запит успішний
    HalfOpen --> Open : Наступний запит невдалий
```

- **Клієнтський Fallback:** Коли Запобіжник переходить у стан `OPEN`, система миттєво повертає клієнту резервну відповідь: *"Наразі система оновлює дані про місця. Ваш запит додано в чергу"*, не чекаючи мережевих таймаутів.

**Математичне обґрунтування (Exponential Backoff with Jitter):**

Для запитів, переведених у чергу, повторні спроби обчислюються за формулою бекофу з джиттером. Це усуває проблему *Thundering Herd* (одночасного шквалу запитів):

$$t_{retry} = \text{random}(0, \min(C \cdot 2^n, t_{max}))$$

Де $n$ — номер спроби, $C$ — базова затримка (напр., 100мс), $t_{max}$ — максимальний час очікування. Джиттер (`random`) рівномірно розмазує навантаження у часі.

---

### 3.2. Розподілені транзакції (Saga Pattern)

**Сценарій Б: Невдача в обробці платежу (Payment Failure)**

Оскільки ми не можемо використовувати класичні ACID-транзакції між розрізненими базами Booking та Payment, ми застосовуємо патерн **Saga (на базі Choreography)**.

- **Проблема:** Місце тимчасово зарезервоване (Soft Lock), але зовнішній банківський шлюз відхиляє транзакцію (наприклад, через брак коштів на картці або досягнення лімітів банку).
- **Рішення (Компенсаційні транзакції):** Якщо платіж відхилено, `Payment Service` публікує в шину подію `PaymentFailed`. Сервіси бронювання та інвентарю вичитують її і автоматично запускають зворотний процес (компенсацію): скасовують статус бронювання та повертають місце в пул доступних квитків.

**Візуалізація компенсаційної транзакції (Saga Flow):**

```mermaid
sequenceDiagram
    autonumber
    participant UI as Клієнт
    participant Booking as Booking Service
    participant Kafka as Event Bus (Kafka)
    participant Inventory as Inventory Service
    participant Payment as Payment Service

    UI->>Booking: Забронювати квиток
    Booking->>Inventory: Блокувати місце (Soft Lock)
    Inventory-->>Booking: ОК
    Booking->>Kafka: Подія [BookingCreated]
    
    Kafka->>Payment: Вичитує подію, ініціює оплату
    Note over Payment: Банк відхиляє транзакцію
    
    Payment->>Kafka: Подія [PaymentFailed]
    
    %% Компенсація
    rect rgb(255, 235, 238)
        Note over Booking, Inventory: КОМПЕНСАЦІЙНА ТРАНЗАКЦІЯ (Rollback)
        Kafka->>Booking: Вичитує [PaymentFailed]
        Booking->>Inventory: Зняти Soft Lock (Повернути квиток)
        Booking->>Booking: Статус = CANCELED
    end
    
    Kafka->>UI: WebSocket: "Оплата відхилена, місце звільнено"
```

---

### 3.3. Керування асинхронними збоями (Dead Letter Queue)

**Сценарій В: Збій відправки сповіщень (Notification Failure)**

- **Проблема:** Зовнішній SMTP-сервер недоступний. Якщо воркер постійно намагатиметься відправити один і той самий лист, він заблокує обробку інших повідомлень у RabbitMQ (head-of-line blocking).
- **Рішення (Dead Letter Queue - DLQ):** Після 5 невдалих спроб (з використанням бекофу), проблемне повідомлення позначається як "отруйне" (Poison Pill) і автоматично маршрутизується в окрему чергу — **DLQ**. Це дозволяє основній черзі працювати без перебоїв, а інженери (отримавши алерт у Grafana) можуть пізніше проаналізувати DLQ і запустити пакетне перевідправлення листів.

---

### 3.4. Неочевидні аспекти та еволюція (Replaceability)

Стратегія обробки помилок має глибокий вплив на загальну стандартизацію системи:

1. **Ідемпотентність компенсацій:** Вузьким місцем Саги є ризик подвійної компенсації (якщо подія `PaymentFailed` прийшла в шину двічі). Шлях усунення: таблиці `Inventory` повинні підтримувати ідемпотентні ключі (Idempotency Keys). Спроба розблокувати вже розблокований квиток має повертати успіх (200 OK), а не помилку стану.
2. **Вплив на UI (Eventual Consistency):** Фронтенд має бути готовим до парадигми "узгодженості в кінцевому рахунку". Інтерфейс повинен вміти коректно реагувати на асинхронні події (через WebSockets), змінюючи статус квитка з "Обробляється" на "Скасовано" без ручного перезавантаження сторінки користувачем.
3. **Замінність API (Standardized Errors):** Усі сервіси (і Fallback-відповіді Gateway) імплементують єдиний глобальний стандарт помилок **RFC 7807 (Problem Details for HTTP APIs)**. Це означає, що структура JSON-помилки завжди уніфікована: `{"type": "...", "title": "...", "status": 503, "detail": "..."}`. Завдяки цьому ми можемо легко замінити API Gateway або повністю переписати клієнтський додаток — обробники помилок на фронтенді ніколи не зламаються.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **4. Стратегія версіювання API (Booking Service)**

В екосистемі «2BeFlüge» клієнти (мобільні застосунки та веб-сайт) рідко оновлюються синхронно. Особливо це стосується мобільних застосунків, де користувачі можуть місяцями не завантажувати нові версії з App Store / Google Play. 

Щоб гарантувати зворотну сумісність і безперебійну роботу старих клієнтів під час впровадження нових фіч (наприклад, додавання динамічного ціноутворення або страхування багажу), ми застосовуємо **Багаторівневу гібридну стратегію версіювання**.

### 4.1. Гібридний підхід до версіювання (Hybrid Versioning)

Різні типи комунікації в нашій архітектурі вимагають різних підходів до еволюції контрактів:

1. **Зовнішній шар (BFF - Backend for Frontend): Безверсійний GraphQL.**
   - Для мобільних та веб-клієнтів ми використовуємо єдину точку входу через GraphQL на API Gateway. GraphQL за своєю природою є **Versionless** (безверсійним). Замість створення `/v2/` ендпоінтів, ми плавно еволюціонуємо єдину схему: додаємо нові поля (`baggageInsurancePrice`), а старі позначаємо директивою `@deprecated(reason: "Використовуйте нове поле priceDetails")`. 
   - Старі мобільні застосунки продовжують запитувати старі поля, а нові — нові. Це повністю усуває проблему версіювання на стороні клієнта.
2. **Внутрішній шар (API Gateway ➡️ Booking Service): URI Versioning (REST).**
   - Між Gateway та сервісом бронювання використовується класичне версіювання через шлях: `/api/v1/bookings` та `/api/v2/bookings`. Це забезпечує жорсткі контракти та дозволяє легко налаштовувати маршрутизацію (Routing) і кешування на рівні інфраструктури (Nginx/Envoy).
3. **Глибинний шар (Booking ➡️ Inventory): Protobuf Evolution (gRPC).**
   - gRPC і Protocol Buffers мають вбудовану зворотну сумісність. Нові поля додаються з новими порядковими номерами (`int32 insurance_id = 4;`). Старі воркери просто ігноруватимуть невідомі поля, а нові — оброблятимуть їх.

**Візуалізація маршрутизації версій (API Gateway Routing):**

```mermaid
flowchart LR
    Client_Old([Старий<br/>Мобільний Застосунок]) -->|GraphQL Старі поля| GW{{API Gateway}}
    Client_New([Новий<br/>Web або Mobile]) -->|GraphQL Нові поля| GW

    subgraph Kubernetes_Cluster
        GW -->|REST POST /v1/bookings| BookingV1[Booking Pods v1]
        GW -->|REST POST /v2/bookings| BookingV2[Booking Pods v2]

        BookingV1 -.->|Спільний БД| DB[(Booking Database)]
        BookingV2 -.->|Спільний БД| DB
    end

    classDef default fill:#f9f9f9,stroke:#333,stroke-width:1px;
    classDef gateway fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    class GW gateway;
```

---

### 4.2. Керування змінами: Патерн Expand and Contract (Parallel Change)

**Проблема (Вузьке місце):** Підтримка кількох версій API часто призводить до конфліктів на рівні бази даних, якщо схема БД суттєво змінюється у версії `v2`. Зупинка системи для масової міграції даних є неприпустимою (Zero Downtime Requirement).

**Рішення:** Застосування архітектурного патерну **Expand and Contract (Розширення та Звуження)**. Цей патерн розбиває несумісну зміну схеми БД на кілька безпечних, зворотно сумісних кроків.

**Візуалізація безшовної еволюції Бази Даних:**

```mermaid
sequenceDiagram
    autonumber
    participant App as Клієнти (V1 та V2)
    participant API as Booking API
    participant DB as База Даних

    rect rgb(232, 245, 233)
        Note over App, DB: Фаза 1: EXPAND (Розширення)
        API->>DB: DDL: Додавання нових колонок (без видалення старих)
        App->>API: Запити від старих (V1) та нових (V2) клієнтів
        API->>DB: Подвійний запис (Dual Write) у старі та нові колонки
        API-->>DB: Читання виконується зі СТАРИХ колонок
    end

    rect rgb(255, 243, 224)
        Note over App, DB: Фаза 2: MIGRATE (Міграція)
        loop Фонова асинхронна job-а
            API->>DB: Перенесення історичних даних зі старих колонок у нові
        end
        API-->>DB: Переключення Читання на НОВІ колонки
    end

    rect rgb(255, 235, 238)
        Note over App, DB: Фаза 3: CONTRACT (Звуження)
        Note over App: Всі клієнти оновилися до V2 (або V1 відключено)
        API->>DB: DDL: Видалення (Drop) старих колонок
    end
```

---

### 4.3. Життєвий цикл API та Sunsetting (Виведення з експлуатації)

Підтримка старих версій API збільшує технічний борг і витрати на інфраструктуру. Для `Booking Service API` впроваджено математично обґрунтовану політику виведення з експлуатації.

**Математична модель відключення (Traffic Decay Threshold):**

Кількість користувачів старого мобільного застосунку з часом зменшується за законом експоненційного згасання:

$$T_{v1}(t) = T_{initial} \cdot e^{-\lambda t}$$

Де $T_{v1}(t)$ — трафік старої версії у день $t$, $T_{initial}$ — трафік на момент релізу `v2`, а $\lambda$ — коефіцієнт швидкості оновлення застосунку користувачами.

* **Deprecation Phase:** За 6 місяців до відключення `v1`, Gateway додає заголовки `Deprecation` та `Link`.
* **Sunset Trigger:** Моніторинг Prometheus постійно обчислює інтеграл трафіку. Як тільки $T_{v1}(t) \le 0.01 \cdot (T_{v1} + T_{v2})$ (тобто трафік старого застосунку стає меншим за 1%), система переходить у фазу примусового відключення.
* **Примусове відключення (Sunset):** `v1` відключається, і Gateway повертає статус `HTTP 410 Gone` із вказівкою оновити мобільний застосунок.

---

### 4.4. Неочевидні аспекти та еволюція (Replaceability)

1. **API Gateway як Anti-Corruption Layer (Антикорупційний шар):** Щоб уникнути розгортання окремих подів для `v1` та `v2` (що витрачає ресурси), ми делегуємо трансляцію версій самому API Gateway. Якщо `v2` вимагає нового обов'язкового поля `currency`, а `v1` його не надсилає, Gateway перехоплює запит `v1`, автоматично додає `"currency": "EUR"` і маршрутизує його на сучасний бекенд `v2`. Це дозволяє **вимкнути старі мікросервіси фізично, зберігаючи віртуальну підтримку старого API**.
2. **Вплив на кешування (CDN):** При використанні URI-версіювання (`/v1/` та `/v2/`) кеш на рівні CDN не інвалідується і не конфліктує, оскільки шлях розглядається як абсолютно унікальний URL. Це зберігає високу продуктивність при паралельному існуванні кількох версій.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **5. Стратегія інтеграції нового сервісу рекомендацій (ML Recommendations)**

Впровадження Machine Learning (ML) сервісу рекомендацій для платформи «2BeFlüge» кардинально відрізняється від інтеграції класичних транзакційних сервісів. ML-моделі вимагають великих обсягів історичних даних, мають високу затримку (Latency) при інференсі (передбаченні) і не повинні впливати на критичний шлях (Core Booking Flow) системи.

Для забезпечення максимальної ізоляції та продуктивності ми застосовуємо **Асинхронну подієву архітектуру (Event-Driven Architecture)** з використанням патерну **CQRS** (Command Query Responsibility Segregation).

### 5.1. Архітектурний патерн: Event-Driven Ingestion & O(1) Retrieval

Прямі синхронні запити від `Booking Service` до `Recommendation Service` є антипатерном, оскільки збій або повільна робота ML-сервісу заблокує продаж квитків.

**Рішення:**
1. **Збір даних (Ingestion):** Сервіси пошуку та бронювання діють як *Producers*. Вони просто публікують події (`UserSearchedFlight`, `BookingCompleted`) у топіки **Apache Kafka** за принципом "Fire-and-Forget".
2. **Обробка (Processing):** Воркери сервісу рекомендацій підписані на ці топіки. Вони асинхронно вичитують дані, оновлюють профілі користувачів (Feature Store) та запускають фонові ML-алгоритми (наприклад, колаборативну фільтрацію).
3. **Видача (Serving):** Готові рекомендації (пре-калькульовані результати) записуються у надшвидку in-memory базу даних **Redis**. Коли Мобільний Застосунок або веб-сайт запитує рекомендації, API миттєво віддає їх з Redis за час $O(1)$.

**Візуалізація інтеграції (Event-Driven ML Ecosystem):**

```mermaid
flowchart TD
    Client([Мобільний Застосунок / Web]) -->|1. Пошук / Бронювання| GW{{API Gateway}}
    GW -->|REST / gRPC| Core[Core Services: Search, Booking]

    Core -->|2. Публікація події| Kafka{Apache Kafka Event Bus}

    subgraph ML_Ecosystem
        Kafka -->|3. Асинхронне споживання| ML_Worker[ML Ingestion Worker]
        ML_Worker -->|4. Навчання та передбачення| Model[(ML Model / Feature Store)]
        Model -->|5. Запис готових рекомендацій| Redis[(Redis Fast Cache)]
    end

    Client -->|6. Запит 'Рекомендовано вам'| GW
    GW -->|gRPC| RecAPI[Recommendation API]
    RecAPI -->|Читання за 2 мс| Redis

    classDef default fill:#f9f9f9,stroke:#333,stroke-width:1px;
    classDef broker fill:#fff3e0,stroke:#e65100,stroke-width:2px;
    class Kafka broker;
```

---

### 5.2. Вирішення вузьких місць: Проблема «Холодного старту» та Data Lag

**Вузьке місце:** ML-моделі страждають від проблеми "Холодного старту" (Cold Start) — коли новий користувач щойно встановив Мобільний Застосунок, для нього ще немає історії пошуку, тому система не знає, що рекомендувати.
**Шлях усунення:** Гібридний інференс. Якщо в Redis немає персональних рекомендацій для конкретного `user_id`, `Recommendation API` використовує Fallback-стратегію: повертає глобальний топ популярних напрямків (Global Trending Flights), який перераховується раз на годину.

**Вузьке місце:** Відставання Kafka (Lag) під час пікових навантажень (наприклад, Чорна П'ятниця).
**Шлях усунення:** Топіки Kafka шардуються (Partitioning) за ключем `user_id`. Це гарантує строгий порядок обробки подій для одного користувача і дозволяє горизонтально масштабувати ML-воркери, просто додаючи нові поди в Kubernetes.

---

### 5.3. Замінність та Еволюція: A/B Тестування (Multi-Armed Bandit)

Сфера Machine Learning розвивається швидко. Сьогодні ми використовуємо Python та TensorFlow, а завтра можемо перейти на PyTorch або взагалі змінити алгоритм на LLM. Архітектура повинна підтримувати непомітну заміну (Replaceability).

Щоб довести, що нова модель "B" краща за поточну "A" (приносить більше кліків і продажів), ми інтегруємо патерн маршрутизації **Epsilon-Greedy (Багаторукий бандит)** безпосередньо в Recommendation API:

$$P(a) = \begin{cases} 1 - \epsilon + \frac{\epsilon}{N}, & \text{якщо } a \text{ — модель з найвищим CTR} \\ \frac{\epsilon}{N}, & \text{для інших експериментальних моделей} \end{cases}$$

Де $P(a)$ — ймовірність того, що клієнт отримає рекомендації від конкретної ML-моделі $a$. Більшість трафіку ($1 - \epsilon$) отримує перевірена модель, але частина трафіку ($\epsilon$) завжди направляється на нові/експериментальні моделі для безперервного тестування гіпотез без ризику для загальної конверсії.

---

### 5.4. Неочевидні аспекти: Ізоляція збоїв (Bulkhead Pattern)

1. **М'яка деградація UI (Graceful Degradation):** Вплив сервісу рекомендацій на загальну систему суворо обмежений патерном *Bulkhead (Перебірки, як на кораблях)*. Якщо `Recommendation API` падає або відповідає довше 200 мс, Мобільний Застосунок розроблений так, щоб просто **не відображати блок "Рекомендовані рейси"**. Користувач не побачить жодної помилки і зможе безперешкодно завершити свій основний флоу (знайти та купити квиток).
2. **Незалежність технологічного стеку:** Завдяки тому, що комунікація відбувається виключно через брокер повідомлень (Kafka) та швидкий кеш (Redis), команда Data Science повністю відв'язана від бекенд-інженерів. Вони можуть переписувати свої мікросервіси на будь-якій мові (Python, Go, C++), не вносячи жодних змін у код `Booking Service` або API Gateway.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **6. Практична симуляція: Моделювання Флеш-розпродажу (Load Spikes & Auto-scaling)**

Щоб довести життєздатність асинхронної архітектури (Розділи 1 та 2), ми змоделюємо сценарій екстремального навантаження — старт акції "Чорна П'ятниця".

**Синтетична симуляція покаже:**

1. **Нормальний режим:** Система стабільно обробляє ~100 запитів на хвилину, тримаючи мінімум серверів для економії ресурсів.
2. **Вхідний трафік (Incoming Requests):** Раптовий стрибок до 4500+ запитів на хвилину від веб-клієнтів та **мобільного застосунку**.
3. **Буферизація (Queue Size):** Як RabbitMQ приймає удар на себе, зберігаючи запити, щоб клієнти не отримали помилку "Timeout".
4. **Автомасштабування (HPA Capacity):** Як Kubernetes динамічно додає нові поди (зелена лінія) у відповідь на зростання черги, а після закінчення розпродажу — зменшує їх (Scale Down) для оптимізації витрат на хмару.

In [6]:
# Симуляція даних - Створення анімації Plotly: Флеш-розпродаж
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import base64
from IPython.display import HTML, display
import warnings

warnings.filterwarnings('ignore')

def simulate_flash_sale(minutes=60, spike_start=15, spike_end=25):
    time_index = [datetime(2026, 11, 27, 10, 0) + timedelta(minutes=i) for i in range(minutes)]
    incoming_requests, worker_capacity, queue_size = [], [], []
    current_queue, current_capacity = 0, 200

    for i in range(minutes):
        if spike_start <= i <= spike_end:
            incoming = int(np.random.normal(4500, 500))
        else:
            incoming = int(np.random.normal(100, 20))
        
        incoming = max(0, incoming)
        incoming_requests.append(incoming)

        if current_queue > 1000 and current_capacity < 3000:
            current_capacity += 600
        elif current_queue < 100 and current_capacity > 200:
            current_capacity -= 200

        worker_capacity.append(current_capacity)
        current_queue += incoming
        processed = min(current_queue, current_capacity)
        current_queue -= processed
        queue_size.append(current_queue)

    return pd.DataFrame({
        'Time': time_index, 'Incoming': incoming_requests,
        'Capacity': worker_capacity, 'Queue_Size': queue_size
    })

df_sim = simulate_flash_sale()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(
    x=[df_sim['Time'].iloc[0]], y=[df_sim['Incoming'].iloc[0]], 
    name="Вхідні запити (Трафік)", marker_color='rgba(54, 162, 235, 0.7)'
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=[df_sim['Time'].iloc[0]], y=[df_sim['Capacity'].iloc[0]], 
    name="Пропускна здатність системи (HPA)", line=dict(color='#00ff00', width=3, dash='dash')
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=[df_sim['Time'].iloc[0]], y=[df_sim['Queue_Size'].iloc[0]], 
    name="Накопичення в RabbitMQ (Черга)", line=dict(color='#ff0033', width=2),
    fill='tozeroy', fillcolor='rgba(255, 0, 51, 0.3)'
), secondary_y=True)

frames = [go.Frame(
    data=[
        go.Bar(x=df_sim['Time'].iloc[:i], y=df_sim['Incoming'].iloc[:i]),
        go.Scatter(x=df_sim['Time'].iloc[:i], y=df_sim['Capacity'].iloc[:i]),
        go.Scatter(x=df_sim['Time'].iloc[:i], y=df_sim['Queue_Size'].iloc[:i])
    ],
    name=str(i)
) for i in range(1, len(df_sim) + 1)]
fig.frames = frames

max_y1 = max(df_sim['Incoming'].max(), df_sim['Capacity'].max()) * 1.05
max_y2 = df_sim['Queue_Size'].max() * 1.05

animation_time_second = 15
frame_duration = int((animation_time_second * 1000) / len(df_sim))

sliders = [dict(
    steps=[dict(method='animate', 
                args=[[str(i)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))],
                label=df_sim['Time'].dt.strftime('%H:%M').iloc[i-1]) for i in range(1, len(df_sim) + 1)],
    active=0,
    transition=dict(duration=0),
    x=0, y=-0.1,
    currentvalue=dict(font=dict(size=14, color="#36a2eb"), prefix="Час: ", visible=True, xanchor="right"),
    len=1.0
)]

fig.update_layout(
    title="<b>Симуляція 2BeFlüge:</b> Поведінка системи під час Флеш-розпродажу",
    template="plotly_dark",
    hovermode="x unified",
    hoverlabel=dict(namelength=-1), 
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    xaxis=dict(range=[df_sim['Time'].min(), df_sim['Time'].max()], title="Час"),
    yaxis=dict(range=[0, max_y1], title="Запити / хв"),
    yaxis2=dict(range=[0, max_y2], title="Повідомлень у черзі", showgrid=False),
    sliders=sliders,
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.02, y=1.15,
        xanchor="left", yanchor="top",
        direction="right",
        pad={"r": 15, "t": 25},
        buttons=[
            dict(label=f"▶ Старт ({animation_time_second} сек)", method="animate",
                 args=[None, {"frame": {"duration": frame_duration, "redraw": True}, "fromcurrent": True}]),
            dict(label="⏸ Пауза", method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
        ]
    )]
)

plot_html = fig.to_html(include_plotlyjs='cdn', full_html=True)
b64_chart = base64.b64encode(plot_html.encode()).decode()

download_link = f'<div style="text-align: center; margin-bottom: 15px;"><a href="data:text/html;base64,{b64_chart}" download="2BeFluge_flash_sale.html" style="color:rgb(54, 162, 235); text-decoration: none; font-size: 16px; font-weight: bold; padding: 10px 20px; border: 1px solid rgb(54, 162, 235); border-radius: 6px; background-color: rgba(54, 162, 235, 0.1);">✈️ Завантажити інтерактивний 2BeFlüge</a></div>'
iframe_display = f'<iframe src="data:text/html;base64,{b64_chart}" width="100%" height="650px" frameborder="0"></iframe>'

display(HTML(download_link))
display(HTML(iframe_display))

---

### 6.1. Практична верифікація стійкості (Chaos Engineering)

Оскільки архітектура **2BeFlüge** закладає потужні теоретичні механізми відмовостійкості (Circuit Breaker, Bulkhead, Saga, DLQ), критично важливо гарантувати їхню безумовну працездатність у реальних "бойових" умовах. Для переходу від теоретичної надійності до доведеної практики впроваджується методологія **Chaos Engineering**.

Для регулярного стрес-тестування інфраструктури (яка базується на Kubernetes) використовується інструмент **Chaos Mesh**. Він дозволяє автоматизовано імітувати відмови на різних рівнях системи під час CI/CD пайплайнів або на staging-середовищі.

**Ключові сценарії хаос-тестування (Chaos Experiments):**
1. **Network Chaos (Імітація мережевих затримок):** Штучне введення затримки (latency) у 5+ секунд між API Gateway та внутрішніми мікросервісами.
   * *Очікуваний результат:* Коректне спрацювання патерну **Circuit Breaker** (перехід у стан Open) та швидке повернення клієнту (веб-клієнт або **мобільний застосунок**) форматованої помилки за стандартом RFC 7807 замість каскадного зависання пулу потоків.
2. **Pod/Node Kill (Раптова зупинка вузлів бази даних або сервісу):** Примусове знищення подів `Payment Service` прямо під час обробки транзакцій.
   * *Очікуваний результат:* Автоматична ініціалізація компенсаційних транзакцій (**Saga Pattern**), які безпомилково скасовують попереднє бронювання місця в `Inventory Service`, гарантуючи консистентність даних без ручного втручання.
3. **Stress Chaos (Вичерпання ресурсів):** Штучне перевантаження CPU/Memory на вузлах сервісу пошуку квитків.
   * *Очікуваний результат:* Активація механізмів **Backpressure** на рівні повідомлень та автоматичне спрацювання **Horizontal Pod Autoscaler (HPA)** для розгортання нових реплік сервісу.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **Загальні висновки (Executive Summary)**

Спроєктована архітектура платформи **2BeFlüge** демонструє комплексний підхід до вирішення проблем розподілених систем, гарантуючи:

1. **Відмовостійкість (Resilience):** Архітектура виключає наявність єдиної точки відмови (SPOF). Крім того, платформа надійно захищена від каскадних відмов завдяки патернам Circuit Breaker, Bulkhead та DLQ.
2. **Еластичність (Elasticity):** Динамічне масштабування (HPA) дозволяє системі розширюватися саме тоді, коли є загроза переповнення, і звужуватися для економії коштів хмарної інфраструктури, коли трафік стабілізується.
3. **Консистентність (Consistency):** Впровадження компенсаційних транзакцій (Saga) та ключів ідемпотентності гарантує цілісність фінансових та інвентарних даних без використання повільних розподілених блокувань.
4. **Еволюційність (Replaceability):** Завдяки застосуванню патернів Anti-Corruption Layer на API Gateway, гібридному версіюванню та стандартизації помилок (RFC 7807), будь-який мікросервіс або базу даних можна замінити чи оновити без простою (Zero Downtime) та без порушення роботи існуючих клієнтів (вебу та **мобільних застосунків**).
5. **Безперервна верифікація (Continuous Validation):** Інтеграція практик **Chaos Engineering** (через Chaos Mesh) дозволяє на практиці перевірити та довести працездатність усіх закладених механізмів в умовах контрольованих збоїв, перетворюючи "паперову" архітектуру на перевірене enterprise-рішення.

Даний HLD-документ (High-Level Design) повністю покриває технічні вимоги до системи та готовий до етапу низькорівневого проєктування (LLD) та імплементації.